In [ ]:
import re
from typing import Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.cm as cm
from multi_llm_debate.analysis.correct_rate_by_round import (
    calculate_correct_rate_by_round,
)
from multi_llm_debate.analysis.calculate_task_accuracy import analyze_task_accuracy
from multi_llm_debate.run.bool_q.utils import extract_bool_answer

# ==========================
# 1. CONFIGURATION
# ==========================
DATA_PATH = Path("../output/bool_q/processed_data.csv")
bool_q_path = Path("../datasets/bool_q")
MODEL_DIR_PATH = Path("../data/bool_q")

# We only want directories whose parentheses-based digit sum == 6
TARGET_MODEL_COUNT = 11

# Maximum round number for your correct_rate_by_round function
MAX_ROUND_NUMBER = 10


# ==========================
# 2. HELPER FUNCTIONS
# ==========================
def get_total_model_count(dir_name: str) -> int:
    """Parses folder name to sum up the numeric values found in parentheses.
    
    Args:
        dir_name: Model directory name, e.g., 'llama3(3)+mistral(3)'
        
    Returns:
        Sum of all numbers found in parentheses
    """
    matches = re.findall(r"\((\d+)\)", dir_name)
    return sum(int(m) for m in matches)


def create_plot(
    accuracies_by_round: Dict[float, Dict[str, np.ndarray]], 
    model_name: str
) -> None:
    """Plots lines for each accuracy value and metric type.
    
    Args:
        accuracies_by_round: Dictionary mapping accuracy values to a dict of
            metrics with their corresponding values.
        model_name: Name of the model for the plot title.
    """
    # Sort the dictionary items by the accuracy value (the dictionary key)
    sorted_items = sorted(accuracies_by_round.items(), key=lambda x: x[0])
    
    # Use a color map and extract the list of colors
    colors = plt.cm.get_cmap('tab20')
    color_list = colors(np.linspace(0, 1, len(sorted_items)))

    plt.figure(figsize=(10, 6))
    
    # Define line styles for different metrics
    line_styles = {
        'absolute': '-',       # solid line
        'majority_vote': '--'  # dashed line
    }
    
    legend_handles = []

    # Plot a line for each unique accuracy value and metric
    for idx, (accuracy, metrics_dict) in enumerate(sorted_items):
        for metric_name, values in metrics_dict.items():
            # Skip empty arrays or None values
            if values is None or len(values) == 0:
                continue
                
            # Create x-axis values matching the length of values
            rounds = np.arange(len(values))
            
            # Only plot if both rounds and values have the same length
            if len(rounds) > 0 and len(rounds) == len(values):
                # Assign a unique color for each line based on accuracy and metric
                color = color_list[idx % len(color_list)]  # Ensures color wrapping if more than 'tab20' colors are needed
                
                line, = plt.plot(
                    rounds,
                    values,
                    color=color,
                    linestyle=line_styles.get(metric_name, '-'),
                    linewidth=2,
                    label=f"Acc={accuracy:.2f} ({metric_name})"
                )
                legend_handles.append(line)

    # Title and labels
    plt.title(f'Accuracy by Round: {model_name}', pad=15)
    plt.xlabel('Round Number')
    plt.ylabel('Correct Rate')
    plt.grid(True, linestyle='--', alpha=0.7)
    
    if legend_handles:
        plt.legend(handles=legend_handles)

    # Set y-axis limits and ticks
    plt.ylim(0, 1)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xticks(range(min(11, MAX_ROUND_NUMBER + 1)))

    plt.tight_layout()
    plt.show()



def process_model(model_dir: Path) -> None:
    """Process model data and create visualizations.
    
    Args:
        model_dir: Path to the model directory containing debate data.
    
    This function:
    1) Analyzes accuracy
    2) Calculates correct rates for each unique accuracy value
    3) Prints the percentage of tasks for each accuracy value
    4) Plots the results for both absolute and majority vote metrics
    """
    model_name = model_dir.name
    print(f"\nProcessing model: {model_name}")

    # 1) Analyze accuracy
    result_df = analyze_task_accuracy(
        model_dir=model_dir,
        dataframe=df,
        extract_fn=extract_bool_answer,
    )

    # 2) Get all unique accuracy values from the result dataframe
    unique_accuracies = result_df['accuracy'].unique()

    # 3) Create a dictionary to store the metrics by round for each accuracy
    accuracies_by_round = {
        accuracy: {'absolute': None, 'majority_vote': None} 
        for accuracy in unique_accuracies
    }

    length = len(result_df)
    # 4) For each unique accuracy value, calculate metrics by round
    for accuracy in unique_accuracies:
        if accuracy < 0:
            continue
            
        # Filter tasks by accuracy
        filtered_df = result_df[result_df['accuracy'] == accuracy]
        
        # Calculate and print the percentage of tasks with this accuracy
        accuracy_percentage = (len(filtered_df) / length) * 100
        print(f"Accuracy = {accuracy:.2f}: {accuracy_percentage:.2f}% of total tasks")

        try:
            # Calculate correct rates for this accuracy
            cr_filtered_df = calculate_correct_rate_by_round(
                filtered_df, 
                model_dir, 
                max_round_number=MAX_ROUND_NUMBER, 
                extract_func=extract_bool_answer
            )
            
            # Check if we have results for the metrics
            absolute_rows = cr_filtered_df[cr_filtered_df['metric'] == 'absolute']
            if not absolute_rows.empty:
                accuracies_by_round[accuracy]['absolute'] = absolute_rows.iloc[0, 2:].values
                
            majority_rows = cr_filtered_df[cr_filtered_df['metric'] == 'majority']
            if not majority_rows.empty:
                accuracies_by_round[accuracy]['majority'] = majority_rows.iloc[0, 2:].values
                
        except Exception as e:
            print(f"Error processing accuracy {accuracy}: {e}")
            continue

    # 5) Create the plot
    create_plot(accuracies_by_round, model_name)


# ==========================
# 3. MAIN SCRIPT
# ==========================
if __name__ == "__main__":
    from multi_llm_debate.run.bool_q.utils import process_bool_q_df
    from multi_llm_debate.utils.download_dataset import load_save_dataset_df
    import os
    
    # Load the main data
    if not DATA_PATH.exists():
        os.makedirs(DATA_PATH.parent, exist_ok=True)
        dataset = load_save_dataset_df(
            dataset_name="google/boolq",
            dataset_path=bool_q_path,
            force_download=False,
        )
        df = process_bool_q_df(dataset)
        df.to_csv(DATA_PATH, index=False)
    else:
        df = pd.read_csv(DATA_PATH)
        
    print("DF columns:", df.columns)
    
    # Get all model directories
    all_model_dirs = list(MODEL_DIR_PATH.glob('*'))

    # Filter directories by total model count == TARGET_MODEL_COUNT
    filtered_model_dirs = [
        d for d in all_model_dirs
        if get_total_model_count(d.name) == TARGET_MODEL_COUNT
    ]

    print(f"Filtered directories (sum of parentheses == {TARGET_MODEL_COUNT}):")
    for d in filtered_model_dirs:
        print("  -", d.name)

    # Process each filtered directory
    for model_dir in filtered_model_dirs:
        process_model(model_dir)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from multi_llm_debate.analysis.calculate_correct_rate_distribution import (
    calculate_correct_rate_distribution_for_round_n,
)
from multi_llm_debate.run.bool_q.utils import extract_bool_answer
from multi_llm_debate.analysis.utils import compare_bool
import pandas as pd
import seaborn as sns
import logging

from typing import Dict, List, Tuple

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

def process_distribution_data(
    result_df: pd.DataFrame,
    round_number: int,
) -> Dict[str, float]:
    """Process distribution data to get percentages for each bin.
    
    Args:
        result_df: DataFrame with distribution data.
        round_number: The round number being processed.
        
    Returns:
        Dictionary mapping bin labels to percentages.
    """
    # Get bin columns (those that are digits)
    bin_columns = [col for col in result_df.columns if col.isdigit()]
    bin_columns.sort(key=int)  # Sort numerically
    
    if not bin_columns or result_df.empty:
        logger.warning(f"No bins found for round {round_number}")
        return {}
    
    # Count tasks and calculate percentages
    task_count = len(result_df)
    bin_sums = result_df[bin_columns].sum()
    bin_percentages = (bin_sums / task_count * 100).to_dict()
    
    return bin_percentages

def plot_round_distribution(
    bin_percentages: Dict[str, float],
    round_number: int,
    output_dir: Path,
    show_plot: bool = False,
) -> None:
    """Create and save a plot of the correct rate distribution for a round.
    
    Args:
        bin_percentages: Dictionary mapping bin labels to percentage values.
        round_number: The round number being visualized.
        output_dir: Directory where the plot should be saved.
        show_plot: Whether to display the plot interactively.
    """
    if not bin_percentages:
        logger.warning(f"No data to plot for round {round_number}")
        return
    
    plt.figure(figsize=(10, 6))
    
    # Sort bins numerically
    bins = [int(b) for b in sorted(bin_percentages.keys(), key=int)]
    values = [bin_percentages[str(b)] for b in bins]
    
    # Create bar chart
    bars = plt.bar(bins, values)
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2.,
            height + 1,
            f'{height:.1f}%',
            ha='center',
            fontsize=9,
        )
    
    # Set chart attributes
    plt.title(f'Round {round_number}: Distribution of Correct Agents', 
             fontsize=14)
    plt.xlabel('Number of Correct Agents', fontsize=12)
    plt.ylabel('Percentage of Tasks (%)', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.ylim(0, max(values) * 1.2)  # Add some headroom for labels
    
    # Save the plot
    output_path = output_dir / f"round_{round_number}_distribution.png"
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    
    if show_plot:
        plt.show()
    plt.close()
    
    logger.info(f"Saved plot for round {round_number} to {output_path}")

def create_heatmap(
    all_distributions: List[Tuple[int, Dict[str, float]]],
    output_dir: Path,
    show_plot: bool = False,
) -> None:
    """Create a heatmap showing the evolution of distributions across rounds.
    
    Args:
        all_distributions: List of (round_number, bin_percentages) tuples.
        output_dir: Directory where the plot should be saved.
        show_plot: Whether to display the plot interactively.
    """
    if not all_distributions:
        logger.warning("No data to create heatmap")
        return
    
    # Create a DataFrame from the collected data
    data = []
    for round_num, bin_percentages in all_distributions:
        for bin_label, percentage in bin_percentages.items():
            data.append({
                'Round': round_num,
                'Correct Agents': int(bin_label),
                'Percentage': percentage
            })
    
    df = pd.DataFrame(data)
    
    # Create pivot table for heatmap
    pivot_df = df.pivot(
        index='Round', 
        columns='Correct Agents', 
        values='Percentage'
    ).fillna(0)
    
    # Create heatmap plot
    plt.figure(figsize=(12, 8))
    ax = sns.heatmap(
        pivot_df,
        annot=True,
        fmt=".1f",
        cmap="YlGnBu",
        linewidths=0.5,
        cbar_kws={'label': 'Percentage of Tasks (%)'}
    )
    
    plt.title('Evolution of Correct Agent Distribution Across Rounds', 
             fontsize=16)
    plt.tight_layout()
    
    # Save the heatmap
    output_path = output_dir / "correct_agent_distribution_heatmap.png"
    plt.savefig(output_path, dpi=300)
    
    if show_plot:
        plt.show()
    plt.close()
    
    logger.info(f"Saved heatmap to {output_path}")

def main(
    data_path: Path,
    model_dir: Path,
    output_dir: Path,
    max_rounds: int = 6,
    show_plots: bool = False,
) -> None:
    """Run the visualization process for correct rate distributions.
    
    Args:
        data_path: Path to the CSV file with task data.
        model_dir: Directory containing model output data.
        output_dir: Directory where output plots should be saved.
        max_rounds: Maximum number of rounds to process.
        show_plots: Whether to display plots interactively.
    """
    # Create output directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # Load answer data (contains correct labels)
        df_answers = pd.read_csv(data_path)
        # Ensure ID column is numeric
        df_answers["id"] = pd.to_numeric(df_answers["id"], errors="coerce")
        df_answers.dropna(subset=["id"], inplace=True)
        df_answers["id"] = df_answers["id"].astype(int)
        logger.info(f"Loaded answer data from {data_path}")
        
        # Load debate data
        debates_path = model_dir / "debate_rounds.csv"
        df_debates = pd.read_csv(debates_path)
        # Ensure numeric columns
        df_debates["task_id"] = pd.to_numeric(df_debates["task_id"], errors="coerce")
        df_debates["round_number"] = pd.to_numeric(
            df_debates["round_number"], errors="coerce"
        )
        df_debates.dropna(subset=["task_id", "round_number"], inplace=True)
        df_debates["task_id"] = df_debates["task_id"].astype(int)
        df_debates["round_number"] = df_debates["round_number"].astype(int)
        logger.info(f"Loaded debate data from {debates_path}")
        
    except Exception as e:
        logger.error(f"Error loading data: {e}")
        return
    
    all_distributions = []
    
    # Process each round
    for round_number in range(max_rounds):
        logger.info(f"Processing round {round_number}...")
        
        # Calculate distribution with the required parameters
        result_df = calculate_correct_rate_distribution_for_round_n(
            df_answers=df_answers,
            df_debates=df_debates,
            round_number=round_number,
            extract_func=extract_bool_answer,
            compare_func=compare_bool,
        )
        
        # Process and store the distribution data
        bin_percentages = process_distribution_data(result_df, round_number)
        
        if bin_percentages:
            all_distributions.append((round_number, bin_percentages))
            
            # Create individual round plot
            plot_round_distribution(
                bin_percentages,
                round_number,
                output_dir,
                show_plot=show_plots,
            )
    
    # Create summary heatmap visualization
    create_heatmap(all_distributions, output_dir, show_plot=show_plots)
    
    logger.info("Visualization complete!")


if __name__ == "__main__":
    # Define paths
    DATA_PATH = Path("../output/bool_q/processed_data.csv")
    MODEL_DIR_PATH = Path("../data/bool_q/llama3(11)")
    OUTPUT_DIR = Path("../output/visualizations")

    # Run the main function
    main(
        data_path=DATA_PATH,
        model_dir=MODEL_DIR_PATH,
        output_dir=OUTPUT_DIR,
        max_rounds=6,
        show_plots=True,  # Set to True to display plots interactively
    )